# Attention Composition Paths

Multi-hop attention: two-hop virtual attention, path strength,
composition scores, and chain analysis.

## Why This Matters

Path attention composition paths traces specific computational pathways through the network. Path-level analysis reveals how information flows from specific input tokens through specific components to specific output predictions, providing the most detailed view of model computation.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis
- [Conmy et al. (2023) "Towards Automated Circuit Discovery"](https://arxiv.org/abs/2304.14997) — Automated methods for circuit finding via edge pruning

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.attention_composition_paths import (
    two_hop_attention, attention_path_strength,
    composition_score_matrix, attention_chain_strength,
    attention_composition_summary,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

## Two-Hop Attention

Virtual attention = layer2 @ layer1. How different is it from direct?

In [ ]:
result = two_hop_attention(model, tokens, layer1=0, layer2=1)
for p in result['per_head_pair']:
    if abs(p['virtual_direct_similarity']) > 0.3:
        print(f"  H{p['head1']}→H{p['head2']}: sim={p['virtual_direct_similarity']:.4f}")

## Path Strength

Attention from source (pos 0) to target (last pos) across layers.

In [ ]:
result = attention_path_strength(model, tokens, source=0, target=-1)
for p in result['per_layer']:
    print(f"  Layer {p['layer']}: max={p['max_strength']:.4f} (H{p['strongest_head']})")
    for h in p['per_head']:
        bar = '█' * int(h['attention_to_source'] * 40)
        print(f"    H{h['head']}: {h['attention_to_source']:.4f} {bar}")

## Composition Score Matrix

How well do layer1 outputs compose with layer2 queries?

In [ ]:
for l1 in range(3):
    l2 = l1 + 1
    result = composition_score_matrix(model, tokens, l1, l2)
    print(f"  Layers {l1}→{l2}: max={result['max_score']:.4f}, mean={result['mean_score']:.4f}")

## Chain Strength

Cumulative path through all layers.

In [ ]:
for src in range(len(tokens)):
    result = attention_chain_strength(model, tokens, source=src, target=-1)
    flag = ' [STRONG]' if result['is_strong_path'] else ''
    print(f"  Source {src}: chain={result['chain_strength']:.6f}, "
          f"per_layer={[f'{s:.4f}' for s in result['max_per_layer']]}{flag}")

## Composition Summary

In [ ]:
result = attention_composition_summary(model, tokens)
for p in result['per_layer_pair']:
    print(f"  Layers {p['layers']}: max_comp={p['max_composition']:.4f}, "
          f"mean_comp={p['mean_composition']:.4f}")